# 🏛️ KG Hà Nội — Bước 3: Build Graph (v2 — đã fix)

**Input:** `kg_markdown/` (từ bước 2.5)
**Output:** `kg/graph.pkl` — NetworkX graph với text embeddings

**Bao gồm:**
1. Parse Markdown → nodes + edges
2. Build NetworkX + entity resolution
3. Merge fragmented nodes (fix 65 components → ít hơn)
4. Encode text embeddings (SentenceBERT)
5. Save graph.pkl
6. Test query (keyword match + PPR)

## 0. Setup

In [ ]:
!pip install sentence-transformers networkx pyyaml -q

import json, re, os, pickle
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

import numpy as np
import networkx as nx
import yaml

# ---- ĐƯỜNG DẪN ----
KG_MD_DIR = "/kaggle/working/kg_markdown"
OUT_DIR = "/kaggle/working/kg"

os.makedirs(OUT_DIR, exist_ok=True)

# Nếu markdown nằm trong input dataset, copy sang working
INPUT_MD = "/kaggle/input/kg_data/kg_markdown"
if Path(INPUT_MD).exists() and not Path(KG_MD_DIR).exists():
    import shutil
    shutil.copytree(INPUT_MD, KG_MD_DIR, dirs_exist_ok=True)
    print(f"✓ Copied {INPUT_MD} → {KG_MD_DIR}")

# Kiểm tra input
for sub in ["landmarks", "food", "dishes", "locations", "other"]:
    p = Path(KG_MD_DIR) / sub
    if p.exists():
        n = len(list(p.glob("*.md")))
        if n > 0:
            print(f"  {sub}/: {n} files")

total = sum(len(list((Path(KG_MD_DIR)/s).glob("*.md")))
            for s in ["landmarks","food","dishes","locations","other"]
            if (Path(KG_MD_DIR)/s).exists())
assert total > 0, f"Không tìm thấy Markdown files ở {KG_MD_DIR}!"
print(f"\n  Tổng: {total} files ✓")

## 0.5 (Tùy chọn) Bổ sung di tích thiếu

Chạy cell này nếu cần thêm di tích mà bước 1 bỏ sót. Sửa/thêm tùy ý.

In [ ]:
# ---- Thêm di tích thiếu ----
EXTRA_LANDMARKS = {
    "Văn_Miếu___Quốc_Tử_Giám.md": """---
name: Văn Miếu – Quốc Tử Giám
aliases: ["Văn Miếu", "Quốc Tử Giám", "Temple of Literature"]
type: Landmark
xây_dựng_năm: "1070"
thuộc_quận: Quận Đống Đa
source: https://vi.wikipedia.org/wiki/Văn_Miếu_–_Quốc_Tử_Giám
---

## Relations
- (Văn Miếu – Quốc Tử Giám, xây_dựng_năm, 1070)
- (Văn Miếu – Quốc Tử Giám, xây_dựng_bởi, Lý Thánh Tông)
- (Văn Miếu – Quốc Tử Giám, triều_đại, Nhà Lý)
- (Văn Miếu – Quốc Tử Giám, thuộc_quận, Quận Đống Đa)
- (Văn Miếu – Quốc Tử Giám, tọa_lạc, Hà Nội)
- (Văn Miếu – Quốc Tử Giám, tọa_lạc, 58 Quốc Tử Giám)
- (Văn Miếu – Quốc Tử Giám, tên_khác, Temple of Literature)
- (Văn Miếu – Quốc Tử Giám, phong_cách_kiến_trúc, Kiến trúc Nho giáo)
- (Văn Miếu – Quốc Tử Giám, tôn_giáo, Nho giáo)
- (Văn Miếu – Quốc Tử Giám, thờ, Khổng Tử)
- (Văn Miếu – Quốc Tử Giám, đặc_điểm, Khuê Văn Các)
- (Văn Miếu – Quốc Tử Giám, đặc_điểm, 82 bia Tiến sĩ)
- (Văn Miếu – Quốc Tử Giám, đặc_điểm, Hồ Thiên Quang)
- (Văn Miếu – Quốc Tử Giám, đặc_điểm, Cổng Tam Quan)
- (Văn Miếu – Quốc Tử Giám, đặc_điểm, Trường đại học đầu tiên của Việt Nam)
- (Văn Miếu – Quốc Tử Giám, sự_kiện, 1076 Lý Nhân Tông lập Quốc Tử Giám)
- (Văn Miếu – Quốc Tử Giám, sự_kiện, 1484 Lê Thánh Tông dựng bia Tiến sĩ đầu tiên)
- (Văn Miếu – Quốc Tử Giám, công_nhận, 82 bia Tiến sĩ được UNESCO công nhận Di sản tư liệu thế giới năm 2010)
- (Văn Miếu – Quốc Tử Giám, xếp_hạng, Di tích quốc gia đặc biệt)
""",
}

landmarks_dir = Path(KG_MD_DIR) / "landmarks"
landmarks_dir.mkdir(parents=True, exist_ok=True)

added = 0
for filename, content in EXTRA_LANDMARKS.items():
    path = landmarks_dir / filename
    if not path.exists():
        path.write_text(content.strip(), encoding="utf-8")
        print(f"  ✓ Thêm: {filename}")
        added += 1
    else:
        print(f"  ⏭ Đã có: {filename}")

print(f"\nThêm {added} di tích")

## 1. Parse Markdown → Nodes + Edges

In [ ]:
def parse_markdown_file(filepath):
    text = filepath.read_text(encoding="utf-8")
    fm_match = re.match(r'^---\s*\n(.*?)\n---', text, re.DOTALL)
    if not fm_match:
        return None, {}, []

    try:
        meta = yaml.safe_load(fm_match.group(1)) or {}
    except yaml.YAMLError:
        meta = {}

    node_id = meta.get("name", filepath.stem)
    aliases = meta.get("aliases", [])
    if isinstance(aliases, str):
        try: aliases = json.loads(aliases)
        except: aliases = [aliases]

    attrs = {
        "type": meta.get("type", "Entity"),
        "aliases": aliases,
        "source": meta.get("source", ""),
        "_file": filepath.name,
    }
    for key in ("xây_dựng_năm", "thuộc_quận", "tọa_lạc", "triều_đại",
                "phong_cách_kiến_trúc", "loại_món", "giá_trung_bình", "tôn_giáo"):
        if key in meta:
            attrs[key] = str(meta[key])

    edges = []
    rel_section = text[fm_match.end():]
    for m in re.finditer(r'-\s*\((.+?),\s*(.+?),\s*(.+?)\)', rel_section):
        edges.append((m.group(1).strip(), m.group(2).strip(), m.group(3).strip()))

    return node_id, attrs, edges


# --- Parse tất cả ---
all_nodes = {}
all_edges = []
alias_map = {}

for sub in ["landmarks", "food", "dishes", "locations", "other"]:
    sub_dir = Path(KG_MD_DIR) / sub
    if not sub_dir.exists(): continue
    for f in sorted(sub_dir.glob("*.md")):
        node_id, attrs, edges = parse_markdown_file(f)
        if node_id is None: continue
        all_nodes[node_id] = attrs
        for alias in attrs.get("aliases", []):
            if alias and alias != node_id:
                alias_map[alias] = node_id
        all_edges.extend(edges)

print(f"Parsed: {len(all_nodes)} nodes, {len(all_edges)} edges, {len(alias_map)} aliases")

## 2. Build NetworkX Graph

In [ ]:
def resolve_entity(name, alias_map):
    return alias_map.get(name, name)

G = nx.DiGraph()

for node_id, attrs in all_nodes.items():
    G.add_node(node_id, **attrs)

relation_counts = defaultdict(int)
for head, rel, tail in all_edges:
    head_r = resolve_entity(head, alias_map)
    tail_r = resolve_entity(tail, alias_map)

    if head_r not in G:
        if rel in ("xây_dựng_năm", "trùng_tu_năm"):
            G.add_node(head_r, type="Year")
        elif rel in ("thuộc_quận", "tọa_lạc"):
            G.add_node(head_r, type="Location")
        elif rel == "triều_đại":
            G.add_node(head_r, type="Dynasty")
        elif rel in ("xây_dựng_bởi", "thiết_kế_bởi"):
            G.add_node(head_r, type="Person")
        elif rel == "phong_cách_kiến_trúc":
            G.add_node(head_r, type="Style")
        elif rel == "loại_món":
            G.add_node(head_r, type="Dish")
        else:
            G.add_node(head_r, type="Entity")

    if tail_r not in G:
        if rel in ("xây_dựng_năm", "trùng_tu_năm"):
            G.add_node(tail_r, type="Year")
        elif rel in ("thuộc_quận", "tọa_lạc"):
            G.add_node(tail_r, type="Location")
        elif rel == "triều_đại":
            G.add_node(tail_r, type="Dynasty")
        elif rel in ("xây_dựng_bởi", "thiết_kế_bởi"):
            G.add_node(tail_r, type="Person")
        elif rel == "phong_cách_kiến_trúc":
            G.add_node(tail_r, type="Style")
        elif rel == "loại_món":
            G.add_node(tail_r, type="Dish")
        else:
            G.add_node(tail_r, type="Entity")

    if head_r != tail_r:
        G.add_edge(head_r, tail_r, relation=rel)
        relation_counts[rel] += 1

for alias, canonical in alias_map.items():
    if alias not in G:
        G.add_node(alias, type="Alias", _alias_of=canonical)
    G.add_edge(alias, canonical, relation="tên_khác")

type_counts = defaultdict(int)
for _, attrs in G.nodes(data=True):
    type_counts[attrs.get("type", "unknown")] += 1

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nNode types:")
for t, c in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {t}: {c}")

print(f"\nTop 10 relations:")
for rel, c in sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {rel}: {c}")

## 3. Merge Fragmented Nodes

In [ ]:
def find_canonical(name, existing_nodes, threshold=0.75):
    name_lower = name.lower().strip()
    best_match, best_score = None, 0
    for node in existing_nodes:
        node_lower = node.lower().strip()
        if name_lower in node_lower or node_lower in name_lower:
            score = 0.9
        else:
            score = SequenceMatcher(None, name_lower, node_lower).ratio()
        if score > best_score:
            best_score = score
            best_match = node
    if best_score >= threshold:
        return best_match
    return None

anchor_types = {"Landmark", "Restaurant", "Dish", "Location"}
anchor_nodes = {n for n in G if G.nodes[n].get("type") in anchor_types}
small_nodes = {n for n in G if n not in anchor_nodes and G.degree(n) <= 2}

merged = 0
for node in list(small_nodes):
    if node not in G: continue
    canonical = find_canonical(node, anchor_nodes)
    if canonical and canonical != node:
        for pred in list(G.predecessors(node)):
            if pred != canonical:
                data = G.edges[pred, node]
                G.add_edge(pred, canonical, **data)
        for succ in list(G.successors(node)):
            if succ != canonical:
                data = G.edges[node, succ]
                G.add_edge(canonical, succ, **data)
        G.remove_node(node)
        merged += 1

print(f"Merged: {merged} nodes")

G_und = G.to_undirected()
components = list(nx.connected_components(G_und))
print(f"Components: {len(components)}")
print(f"Largest: {len(max(components, key=len))} nodes")
small_c = [c for c in components if len(c) < 5]
print(f"Components < 5 nodes: {len(small_c)}")
print(f"\nGraph sau merge: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 4. Encode Text Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded ✓")

# Encode nodes
print("\nEncoding nodes...")
node_texts, node_ids = [], []
for nid in G.nodes:
    attrs = G.nodes[nid]
    parts = [nid]
    for key in ("xây_dựng_năm", "thuộc_quận", "tọa_lạc", "triều_đại",
                "phong_cách_kiến_trúc", "loại_món", "tôn_giáo"):
        if key in attrs:
            parts.append(f"{key}: {attrs[key]}")
    node_texts.append(". ".join(parts))
    node_ids.append(nid)

node_embs = encoder.encode(node_texts, show_progress_bar=True, batch_size=64)
for i, nid in enumerate(node_ids):
    G.nodes[nid]["text_embedding"] = node_embs[i].tolist()
print(f"  {len(node_ids)} nodes ✓")

# Encode edges
print("\nEncoding edges...")
edge_texts, edge_keys = [], []
for u, v, data in G.edges(data=True):
    rel = data.get("relation", "liên_quan")
    edge_texts.append(f"{u} {rel} {v}")
    edge_keys.append((u, v))

edge_embs = encoder.encode(edge_texts, show_progress_bar=True, batch_size=64)
for i, (u, v) in enumerate(edge_keys):
    G.edges[u, v]["text_embedding"] = edge_embs[i].tolist()
print(f"  {len(edge_keys)} edges ✓")

## 5. Save Graph

In [ ]:
graph_path = Path(OUT_DIR) / "graph.pkl"
with open(graph_path, "wb") as f:
    pickle.dump(G, f)
print(f"✓ Saved: {graph_path} ({graph_path.stat().st_size / 1024 / 1024:.1f} MB)")

meta = {
    "num_nodes": G.number_of_nodes(),
    "num_edges": G.number_of_edges(),
    "node_types": dict(type_counts),
    "has_text_embeddings": True,
    "landmarks": [n for n in G if G.nodes[n].get("type") == "Landmark"],
    "restaurants": [n for n in G if G.nodes[n].get("type") == "Restaurant"],
}
meta_path = Path(OUT_DIR) / "meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ Saved: {meta_path}")

## 6. Kiểm tra di tích quan trọng

In [ ]:
must_have = [
    "Văn Miếu", "Chùa Một Cột", "Hoàng thành Thăng Long",
    "Hồ Hoàn Kiếm", "Đền Ngọc Sơn", "Chùa Trấn Quốc",
    "Tháp Rùa", "Cột cờ Hà Nội", "Nhà hát Lớn Hà Nội",
    "Cầu Long Biên", "Lăng Chủ tịch Hồ Chí Minh",
    "Nhà tù Hỏa Lò", "Chùa Hương", "Thành Cổ Loa",
]

landmarks = {n for n in G if G.nodes[n].get("type") == "Landmark"}

for name in must_have:
    found = any(name.lower() in n.lower() for n in landmarks)
    status = "✓" if found else "✗ THIẾU"
    print(f"  {status}  {name}")

# Quick subgraph test
test_node = None
for n in landmarks:
    if "một cột" in n.lower() or "văn miếu" in n.lower():
        test_node = n
        break
if test_node is None and landmarks:
    test_node = list(landmarks)[0]

if test_node:
    print(f"\nTest entity: {test_node}")
    for _, neighbor, data in G.edges(test_node, data=True):
        rel = data.get("relation", "?")
        print(f"  → [{rel}] → {neighbor}")
    ego = nx.ego_graph(G.to_undirected(), test_node, radius=2)
    print(f"\n  2-hop: {ego.number_of_nodes()} nodes, {ego.number_of_edges()} edges")

## 7. Test Query (Keyword Match + PPR)

In [ ]:
def text_query_kg(G, question, encoder, top_n=10, alpha=0.15):
    seed = None
    best_len = 0
    question_lower = question.lower()

    priority_nodes = [n for n in G if G.nodes[n].get("type") in ("Landmark", "Restaurant")]

    for nid in priority_nodes:
        nid_lower = nid.lower()

        # Check 1: tên node nguyên vẹn trong câu hỏi
        if nid_lower in question_lower and len(nid) > best_len:
            seed = nid
            best_len = len(nid)

        # Check 2: tách tên theo dấu gạch (Văn Miếu – Quốc Tử Giám → ["Văn Miếu", "Quốc Tử Giám"])
        name_parts = re.split(r'\s*[–\-]\s*', nid)
        for part in name_parts:
            part = part.strip()
            if len(part) >= 3 and part.lower() in question_lower and len(part) > best_len:
                seed = nid
                best_len = len(part)

        # Check 3: aliases
        for alias in G.nodes[nid].get("aliases", []):
            if len(alias) >= 3 and alias.lower() in question_lower and len(alias) > best_len:
                seed = nid
                best_len = len(alias)

    # Fallback: embedding similarity
    if seed is None:
        q_emb = encoder.encode(question)
        scores = {}
        for nid in priority_nodes:
            if "text_embedding" in G.nodes[nid]:
                emb = np.array(G.nodes[nid]["text_embedding"])
                cos = np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb) + 1e-8)
                scores[nid] = float(cos)
        if scores:
            seed = max(scores, key=scores.get)
            print(f"  Seed (embedding fallback): {seed} (score: {scores[seed]:.3f})")
        else:
            print("  Không tìm được seed!")
            return ""
    else:
        print(f"  Seed (keyword match): {seed}")

    # PPR
    G_und = G.to_undirected()
    try:
        ppr = nx.pagerank(G_und, alpha=alpha,
                          personalization={seed: 1.0}, max_iter=100)
    except:
        ppr = {seed: 1.0}

    top_ppr = sorted(ppr.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # Reasoning paths
    lines = []
    seen = set()
    for target, score in top_ppr:
        if target == seed: continue
        try:
            for path in nx.all_simple_paths(G_und, seed, target, cutoff=2):
                for i in range(len(path) - 1):
                    s, d = path[i], path[i+1]
                    edge = G.get_edge_data(s, d) or G.get_edge_data(d, s) or {}
                    rel = edge.get("relation", "liên_quan")
                    key = tuple(sorted([s, d]) + [rel])
                    if key not in seen:
                        lines.append(f"  {s} → [{rel}] → {d}")
                        seen.add(key)
        except:
            continue

    return "\n".join(lines[:15])


# --- Test ---
test_questions = [
    "Văn Miếu xây dựng năm nào?",
    "Chùa Một Cột ở quận nào?",
    "Quán phở nào gần Hồ Hoàn Kiếm?",
    "Nhà hát Lớn Hà Nội do ai thiết kế?",
    "Cầu Long Biên xây năm nào?",
]

for q in test_questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    print(f"{'='*50}")
    result = text_query_kg(G, q, encoder)
    print(result)

## 8. Download

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/kg_graph", "zip", OUT_DIR)
print(f"✓ /kaggle/working/kg_graph.zip")
# from IPython.display import FileLink
# FileLink("/kaggle/working/kg_graph.zip")